# 00b — Statistiques descriptives par segment

On reprend la base obtenue à la fin du notebook 00 (périmètre final, dossiers joints unifiés, segment SA / SB) et on décrit les variables séparément dans chaque segment, puisque chaque segment aura son propre modèle.

Pour chaque segment :
1. valeurs manquantes ;
2. variables constantes ou presque ;
3. distributions et valeurs extrêmes ;
4. variables qualitatives ;
5. redondances entre variables ;
6. pouvoir discriminant (IV) ;
7. synthèse : statut provisoire de chaque variable avant la discrétisation, et export Excel.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import polars as pl
from utils import binning, exploration

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(80)

ID, TARGET, DATE, SEGMENT = "gdt", "ddefaut_ndb", "datdelhis", "segment"
BASE = Path("data/processed/base_segmentee.parquet")

# Chargement optimisé depuis le fichier Parquet
base = pl.read_parquet(BASE)

# Répartition globale des segments
base.group_by(SEGMENT).agg(
    n=pl.len(), n_defaut=pl.col(TARGET).sum(),
    taux_defaut=pl.col(TARGET).mean()
).sort(SEGMENT)

In [ ]:
valeurs_segments = sorted(base[SEGMENT].unique().to_list())
print("Segments disponibles :", valeurs_segments)

# Sélection du premier segment (Segment 1)
nom_seg1 = valeurs_segments[0]
df_seg1 = base.filter(pl.col(SEGMENT) == nom_seg1)

print(f"Segment analysé : {nom_seg1}")
print(f"Dimensions : {df_seg1.shape[0]} lignes, {df_seg1.shape[1]} colonnes")
print(f"Taux de défaut observé : {df_seg1[TARGET].mean():.2%}")

In [ ]:
EXCLURE = {ID, TARGET, DATE, SEGMENT, "has_engagement", "SelectionProb", "SamplingWeight", "type_PM_PP", "code_marche_v2"}

VARS = [c for c, t in df_seg1.schema.items() if c not in EXCLURE and not t.is_temporal()]
VARS_NUM = [c for c in VARS if df_seg1.schema[c].is_numeric()]
VARS_QUALI = [c for c in VARS if c not in VARS_NUM]

print(f"{len(VARS)} variables analysées : {len(VARS_NUM)} numériques, {len(VARS_QUALI)} qualitatives")

# Analyse des valeurs manquantes

In [ ]:
# 1. Taux de valeurs manquantes par variable
stats_manquants = df_seg1.select(
    [(pl.col(c).null_count() / pl.len()).alias(c) for c in VARS]
).transpose(include_header=True, header_name="variable", column_names=["pct_manquant"])

print("Top 10 variables avec le plus de manquants :")
print(stats_manquants.sort("pct_manquant", descending=True).head(10))

# 2. Statistiques des variables numériques (moyenne, écart-type, quartiles...)
print("\nStatistiques des variables numériques :")
df_seg1.select(VARS_NUM).describe()